In [3]:
"""
Add more as needed
"""
from transformer_lens import HookedTransformer
import torch as t
from torch import Tensor
import einops
import circuitsvis as cv
from jaxtyping import Float, Int
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
import plotly.express as px
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")



In [4]:
"""
Save just a little more time when adding more libraries
"""
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

/var/folders/lx/h48548hj3c1d6b032lzbhqz00000gn/T/ipykernel_33437/2061229548.py:4: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


### Inspecting the Model

In [5]:
prompt = "Mechanistic Interpretability (MI) is the study of reverse engineering neural networks. " \
"Taking an inscrutable stack of matrices where we know that it works, and trying to reverse engineer how it works." \
"And often this inscrutable stack of matrices can be decompiled to a human interpretable algorithm! " \
"In my (highly biased) opinion, this is one of the most exciting research areas in ML."

loss = gpt2_small(prompt, return_type="loss")
logits = gpt2_small(prompt, return_type="logits")



print("Model Loss", loss)
print("Logits shape", logits.shape)
#print("Both", both)

#print(gpt2_small.cfg)

Model Loss tensor(3.6181, grad_fn=<DivBackward0>)
Logits shape torch.Size([1, 84, 50257])


### Converting from Tokens to Strings

In [6]:
print(gpt2_small.to_str_tokens("Keep"))
print(gpt2_small.to_str_tokens(["tokenize", "more", "strings"]))
print(gpt2_small.to_tokens(["tokenize", "more", "strings"]))
print(gpt2_small.to_string(t.tensor([30001, 1096])))
print(gpt2_small.to_string([30001, 1096]))


"""
We'll now determine how many tokens the model guessed correctly
We're quite litrally checking what the model 
Don't be shy to extensively use the ".shape" 
"""

logits = gpt2_small(prompt, return_type="logits")
# The last token can't be compared as it exceed the prompt length
# for the N-th (inside) token provided it'll predict the N+1th token (outside) 
predictions = logits.argmax(-1).squeeze()[:-1]
# The first token isn't predicted from the model 
true_tokens = gpt2_small.to_tokens(prompt).squeeze()[1:]

num_correct = sum(true_tokens==predictions)

print(num_correct.item())
print(len(true_tokens))






['<|endoftext|>', 'Keep']
[['<|endoftext|>', 'token', 'ize'], ['<|endoftext|>', 'more'], ['<|endoftext|>', 'strings']]
tensor([[50256, 30001,  1096],
        [50256,  3549, 50256],
        [50256, 37336, 50256]])
tokenize
tokenize
29
83


### Caching all Activations

Use: `logits cache = model.run_with_cache(tokens)`

We'll also 

In [7]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)

gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

print(gpt2_cache)

"""
Say I wanted to confirm one of the attention heads
"""
q_0 = gpt2_cache['blocks.0.attn.hook_q']
posn, n_heads, d_head = q_0.shape
k_0 = gpt2_cache['blocks.0.attn.hook_k']
attn_score = einops.einsum(q_0, k_0, "posnQ n_heads d_head, posnK n_heads d_head -> n_heads posnQ posnK")
attn_score_scaled = attn_score/(d_head**0.5)
l0_cached = gpt2_cache['blocks.0.attn.hook_pattern']

mask = t.tril(t.ones(posn, posn))
masked = attn_score_scaled.masked_fill(mask==0, -1e9)
probs_0 = masked.softmax(-1)

print(t.allclose(probs_0, l0_cached, atol=1e-4))




ActivationCache with keys ['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_normalized', 'blocks.1.mlp.hook_pre', 'blocks.1.mlp.hook_post', 'blocks.1.hook_mlp_out', 'blocks.1.hook_resid_post', 'blocks.2.hook_re

In [8]:
print(type(gpt2_cache))
attention_pattern = gpt2_cache["pattern", 0]
print(attention_pattern.shape)
gpt2_str_tokens = gpt2_small.to_str_tokens(gpt2_text)

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_patterns(
        tokens=gpt2_str_tokens,
        attention=attention_pattern,
        )
        )


<class 'transformer_lens.ActivationCache.ActivationCache'>
torch.Size([12, 33, 33])
Layer 0 Head Attention Patterns:


In [9]:
import plotly.graph_objects as go

def attention_heatmap(pattern, str_tokens, layer=0):
    """pattern: (n_heads, T, T) for one layer."""
    p = pattern.detach().cpu().numpy()
    labels = [f"{i}:{s}" for i, s in enumerate(str_tokens)]
    n_heads = p.shape[0]

    fig = go.Figure()
    for h in range(n_heads):
        fig.add_trace(go.Heatmap(
            z=p[h], x=labels, y=labels,
            colorscale="Viridis", visible=(h == 0),
            hovertemplate="query %{y}<br>key %{x}<br>attn %{z:.3f}<extra></extra>",
        ))

    fig.update_layout(
        updatemenus=[dict(
            buttons=[dict(label=f"L{layer}H{h}", method="update",
                          args=[{"visible": [i == h for i in range(n_heads)]}])
                     for h in range(n_heads)],
            x=0.0, y=1.1)],
        xaxis_title="key (attended to)",
        yaxis_title="query (attending)",
        width=900, height=900,
        margin=dict(l=120, b=120, t=80),
    )
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=8))
    fig.update_xaxes(tickangle=90, tickfont=dict(size=8))
    return fig

pattern = gpt2_cache["pattern", 0]          # (12, T, T); use [0] first if it has a batch dim
fig = attention_heatmap(pattern, gpt2_str_tokens, layer=0)
fig.show()
fig.write_html("attention_layer0.html")     # standalone file

In [10]:
config = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)

from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [11]:
model = HookedTransformer(config)
pretrained_weights=t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

/var/folders/lx/h48548hj3c1d6b032lzbhqz00000gn/T/ipykernel_33437/4143056761.py:1: DeprecationWarning: HookedTransformer is deprecated and will be removed in 4.0. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics.
  model = HookedTransformer(config)


<All keys matched successfully>

In [12]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

str_tokens = model.to_str_tokens(text)
for layer in range(model.cfg.n_layers):
    attention_pattern = cache["pattern", layer]
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern))

In [13]:
def current_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    scores = t.zeros(n_layers, n_heads)
    for layer in range(n_layers):
        pattern = cache["pattern", layer]
        for head in range(n_heads):
            scores[layer, head] = pattern[head].diagonal(offset=0).mean()

    threshold = scores.mean() + 2*scores.std()
    selected = (scores > threshold).nonzero().tolist()
    return [f"{layer}.{head}" for layer, head in selected]
        


def prev_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    scores = t.zeros(n_layers, n_heads)
    for layer in range(n_layers):
        pattern = cache["pattern", layer]
        for head in range(n_heads):
            scores[layer, head] = pattern[head].diagonal(offset=-1).mean()

    threshold = scores.mean() + 2*scores.std()
    selected = (scores >= threshold).nonzero().tolist()
    return [f"{layer}.{head}" for layer, head in selected]
            


def first_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    scores = t.zeros(n_layers, n_heads)
    for layer in range(n_layers):
        for head in range(n_heads):
            attention_pattern = cache["pattern", layer][head]
            scores[layer, head] = attention_pattern[:, 0].mean()

    threshold = scores.mean() + 2*scores.std()
    selected = (scores >= threshold).nonzero().tolist()
    return [f"{layer}.{head}" for layer, head in selected]


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))

Heads attending to current token  =  0.9, 0.11, 1.6
Heads attending to previous token =  0.7
Heads attending to first token    =  0.3, 1.4, 1.10


## Induction Head

In [24]:
def generate_repeated_token(model: HookedTransformer, seq_len: int, batch_size: int = 1) -> Int[Tensor, "batch_size full_seq_len"]:
    t.manual_seed(0)
    """
    Here we're essentially creating two sentences being repeated for some given length 
    and ensuring that the first token is the BOS or viewed as |<endoftext>|
    You can inspect this by looking at the shapes and content
    """
    prefix = (t.ones(batch_size, 1)*model.tokenizer.bos_token_id).long()
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.int64)
    rep_tokens = t.cat([prefix, rep_tokens_half, rep_tokens_half], dim=-1).to(device)

    return rep_tokens

def run_and_cache_model_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens,
    logits, cache). This function should use the `generate_repeated_tokens` function above.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    rep_tokens = generate_repeated_token(model, seq_len, batch_size)
    rep_logits, rep_cache = model.run_with_cache(rep_tokens)
    return rep_tokens, rep_logits, rep_cache

def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    log_probs = logits.log_softmax(dim=-1)
    # We want to get logprobs[b, s, tokens[b, s+1]], in eindex syntax this looks like:
    next_tokens = tokens[:, 1:].to(log_probs.device)
    correct_log_probs = log_probs[:, :-1].gather(-1, next_tokens.unsqueeze(-1)).squeeze(-1)
    return correct_log_probs


seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")



Performance on the first half: -14.923
Performance on the second half: -6.327


In [ ]:
def induction_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    attn_heads = []
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    for layer in range(n_layers):
        for head in range(n_heads):
            attention_pattern = cache["pattern", layer][head]
            seq_len = (attention_pattern.shape[-1] -1) // 2
            score = attention_pattern.diagonal(-seq_len + 1).mean()
            if score > 0.5:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


print("Induction heads = ", ", ".join(induction_attn_detector(rep_cache)))

for layer in range(model.cfg.n_layers):
    attention_pattern = rep_cache["pattern", layer]
    display(cv.attention.attention_patterns(tokens=rep_str, attention=attention_pattern))


Induction heads =  1.4, 1.10
